## **Evaluation Setup**

In [6]:
a = [{"prompt": "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nyou are a cyber security expert that can detect vulnerable line numbers from the contract as well as vulnerableCode, vulnerabilityReason, potentialSecurityRisk, and fixedCode.\nfind all tiemstamp dependency vulnerabilities  and detect vulnerable line numbers from the contract as well as vulnerableCode, vulnerabilityReason, potentialSecurityRisk, and fixedCode based on the instruction.\n\n1: \n2: \n3: function rescueTokens(uint _pollID) public {\n4: require(isExpired(pollMap[_pollID].revealEndDate));\n5: require(dllMap[msg.sender].contains(_pollID));\n6: \n7: dllMap[msg.sender].remove(_pollID);\n8: emit _TokensRescued(_pollID, msg.sender);\n9: }<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", "predict": "{\"vulnerableLines\": \"\", \"vulnerableCode\": [], \"vulnerabilityReason\": \"\", \"potentialSecurityRisk\": \"\", \"fixedCode\": \"\"}", "label": "{\"vulnerableLines\": \"\", \"vulnerableCode\": [], \"vulnerabilityReason\": \"\", \"potentialSecurityRisk\": \"\", \"fixedCode\": \"\"}"}]

In [8]:
b= [{"prompt": "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nyou are a cyber security expert that can detect vulnerable line numbers from the contract as well as vulnerableCode, vulnerabilityReason, potentialSecurityRisk, and fixedCode.\nfind all tiemstamp dependency vulnerabilities  and detect vulnerable line numbers from the contract as well as vulnerableCode, vulnerabilityReason, potentialSecurityRisk, and fixedCode based on the instruction.\n\n1: address winner_38;\n2: function play_38(uint startTime) public {\n3: if (startTime + (5 * 1 days) == block.timestamp){\n4: winner_38 = msg.sender;}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", "predict": "{\"vulnerableLines\": \"1-4\", \"vulnerableCode\": [\"address winner_38;\", \"function play_38(uint startTime) public {\", \"    if (startTime + (5 * 1 days) == block.timestamp){\", \"        winner_38 = msg.sender;}}\"], \"vulnerabilityReason\": \"The function play_38 uses block.timestamp to compare against a calculated future time. This can lead to exploiters manipulating the block timestamp to gain an unintended advantage.\", \"potentialSecurityRisk\": \"A miner could set the timestamp to match the winning condition, allowing them or a colluding party to win instantly, leading to potential loss of funds for other players.\", \"fixedCode\": \"function play_38(uint startTime) public { require(startTime + (5 * 1 days) <= block.timestamp, 'Game not started or already finished'); winner_38 = msg.sender; }\"}", "label": "{\"vulnerableLines\": \"1-4\", \"vulnerableCode\": [\"address winner_38;\", \"function play_38(uint startTime) public {\", \"    if (startTime + (5 * 1 days) == block.timestamp){\", \"        winner_38 = msg.sender;}}\"], \"vulnerabilityReason\": \"The function play_38 uses block.timestamp in the condition for setting a winner, making it susceptible to manipulation by a miner.\", \"potentialSecurityRisk\": \"By crafting a block with a manipulated timestamp, a miner could unfairly claim the winner's position, leading to loss or unfair advantage.\", \"fixedCode\": \"function play_38(uint startTime) public { require(startTime + (5 * 1 days) <= block.timestamp, 'Game not started or already finished'); winner_38 = msg.sender; }\"}"}
]

In [15]:
print(a[0]['predict'])

{"vulnerableLines": "", "vulnerableCode": [], "vulnerabilityReason": "", "potentialSecurityRisk": "", "fixedCode": ""}


In [12]:
print(b[0]['predict'])

{"vulnerableLines": "1-4", "vulnerableCode": ["address winner_38;", "function play_38(uint startTime) public {", "    if (startTime + (5 * 1 days) == block.timestamp){", "        winner_38 = msg.sender;}}"], "vulnerabilityReason": "The function play_38 uses block.timestamp to compare against a calculated future time. This can lead to exploiters manipulating the block timestamp to gain an unintended advantage.", "potentialSecurityRisk": "A miner could set the timestamp to match the winning condition, allowing them or a colluding party to win instantly, leading to potential loss of funds for other players.", "fixedCode": "function play_38(uint startTime) public { require(startTime + (5 * 1 days) <= block.timestamp, 'Game not started or already finished'); winner_38 = msg.sender; }"}


In [14]:
i=2
print (a[i]['predict'])

{"vulnerableLines": "", "vulnerableCode": [], "vulnerabilityReason": "", "potentialSecurityRisk": "", "fixedCode": ""}


In [15]:
print (a[i]['label'])

{"vulnerableLines": "", "vulnerableCode": [], "vulnerabilityReason": "", "potentialSecurityRisk": "", "fixedCode": ""}


In [22]:
a = []
with open("generated_predictions_all_llama31.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        a.append(json.loads(line))
print(a[12]['predict'])

{"vulnerableLines": "", "vulnerableCode": [], "vulnerabilityReason": "", "potentialSecurityRisk": "", "fixedCode": ""}


In [23]:
b=[]
with open("generated_predictions_all_llama31_2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        b.append(json.loads(line))
print(b[20]['predict'])

{"vulnerableLines": "", "vulnerableCode": [], "vulnerabilityReason": "", "potentialSecurityRisk": "", "fixedCode": ""}


In [19]:
import json
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

In [20]:
a = []
with open("generated_predictions_all_llama31_3.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        a.append(json.loads(line))

def eval_metrics(vul_range):
    if not vul_range or vul_range.strip() == "":
        return set()
    parts = vul_range.split('-')
    if len(parts) == 1:
        return set([int(parts[0])])
    return set(range(int(parts[0]), int(parts[1]) + 1))

def parse_field(field):
    if not field or field.strip() == "":
        return {"vulnerableLines": ""}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {"vulnerableLines": ""}


In [21]:
y_true = []
y_pred = []
results = []

for i, sample in enumerate(a):
    label_dict = parse_field(sample.get("label", ""))
    pred_dict = parse_field(sample.get("predict", ""))

    label_range_str = label_dict.get("vulnerableLines", "")
    pred_range_str = pred_dict.get("vulnerableLines", "")

    label_lines = eval_metrics(label_range_str)
    pred_lines = eval_metrics(pred_range_str)
    # print(label_lines)
    # print(pred_lines)
    # print(label_lines & pred_lines)
    if not label_lines and not pred_lines:
        y_true.append(0)
        y_pred.append(0)
        results.append({"id": i, "result": "TN"})
    elif label_lines and not pred_lines:
        y_true.append(1)
        y_pred.append(0)
        results.append({"id": i, "result": "FN"})
    elif not label_lines and pred_lines:
        y_true.append(0)
        y_pred.append(1)
        results.append({"id": i, "result": "FP"})
    else:
        overlap = label_lines & pred_lines
        if overlap:
            y_true.append(1)
            y_pred.append(1)
            results.append({"id": i, "result": "TP"})
        else:
            y_true.append(1)
            y_pred.append(0)
            results.append({"id": i, "result": "FN (no overlap)"})

In [22]:
#  print(y_true)
# print( y_pred)

In [23]:
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)

print("Evaluation Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

# Optional: Save results
with open("prediction_eval_results_3.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
        #print(r)

Evaluation Metrics:
Accuracy : 0.8489
Precision: 0.9268
Recall   : 0.8034
F1-score : 0.8607


## **Evaluation of Generated Text**

## Using Rouge:

In [20]:
!pip install rouge-score bert-score nltk --quiet


In [98]:
from rouge_score import rouge_scorer
import json

def safe_parse_json(field):
    if not field or field.strip() == "":
        return {}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {}

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

rouge_scores = []

for sample in a[:20]:
    label = safe_parse_json(sample.get("label", ""))
    pred = safe_parse_json(sample.get("predict", ""))

    ref = label.get("vulnerabilityReason", "")
    hyp = pred.get("vulnerabilityReason", "")

    score = rouge.score(ref, hyp)
    rouge_scores.append(score['rougeL'].fmeasure)

print(f"Average ROUGE-L Score: {sum(rouge_scores)/len(rouge_scores):.4f}")


Average ROUGE-L Score: 0.3118


## Using Bleu

In [94]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt')

def compute_bleu(reference, prediction):
    smoothie = SmoothingFunction().method4
    reference_tokens = [nltk.word_tokenize(reference.lower(), preserve_line=True)]
    prediction_tokens = nltk.word_tokenize(prediction.lower(), preserve_line=True)
    return sentence_bleu(reference_tokens, prediction_tokens, smoothing_function=smoothie)

# Handle empty strings and malformed JSON
def safe_parse_json(field):
    if not field or field.strip() == "":
        return {}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {}

# Compute BLEU scores
bleu_scores = []

for sample in a[:20]:
    label = safe_parse_json(sample.get("label", ""))
    pred = safe_parse_json(sample.get("predict", ""))

    ref = label.get("vulnerabilityReason", "")
    hyp = pred.get("vulnerabilityReason", "")

    bleu_scores.append(compute_bleu(ref, hyp))

print(f"Average BLEU Score: {sum(bleu_scores)/len(bleu_scores):.4f}")


Average BLEU Score: 0.1563


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\erfan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


##  Embeding Consistency Score:

In [10]:
!pip install -q sentence-transformers


In [93]:
from sentence_transformers import SentenceTransformer, util
import json

# Load lightweight sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Safe JSON parser
def safe_parse_json(field):
    if not field or field.strip() == "":
        return {}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {}

# Compute semantic similarity between vulnerableCode and vulnerabilityReason
consistency_scores = []

for sample in a[:20]:  # or all: a
    label = safe_parse_json(sample.get("label", ""))
    code_lines = label.get("vulnerableCode", [])
    reason = label.get("vulnerabilityReason", "")

    if not code_lines or not reason.strip():
        continue

    code_text = "\n".join(code_lines)

    code_emb = model.encode(code_text, convert_to_tensor=True)
    reason_emb = model.encode(reason, convert_to_tensor=True)

    score = util.cos_sim(code_emb, reason_emb).item()
    consistency_scores.append(score)

# Report
avg_score = sum(consistency_scores) / len(consistency_scores) if consistency_scores else 0
print(f"Average Consistency Score (Embedding Similarity): {avg_score:.4f}")


Average Consistency Score (Embedding Similarity): 0.4008


## Factual consistency of **"VulnerabilityReason"** of prediction and ground truth:

In [92]:
from sentence_transformers import SentenceTransformer, util
import json

# Load lightweight sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Safe JSON parser
def safe_parse_json(field):
    if not field or field.strip() == "":
        return {}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {}

# Compute semantic similarity between label and predicted vulnerabilityReason
factual_scores = []

for sample in a:  # Use all samples
    label = safe_parse_json(sample.get("label", ""))
    pred = safe_parse_json(sample.get("predict", ""))

    label_reason = label.get("vulnerabilityReason", "")
    pred_reason = pred.get("vulnerabilityReason", "")

    if not label_reason.strip() or not pred_reason.strip():
        continue

    label_emb = model.encode(label_reason, convert_to_tensor=True)
    pred_emb = model.encode(pred_reason, convert_to_tensor=True)

    score = util.cos_sim(label_emb, pred_emb).item()
    factual_scores.append(score)

# Report
avg_score = sum(factual_scores) / len(factual_scores) if factual_scores else 0
print(f"Average Factual Consistency Score (Label ↔ Predict Reason): {avg_score:.4f}")


Average Factual Consistency Score (Label ↔ Predict Reason): 0.7625


## Code-Aware Similarity (VulnerableCode <--> FixedCode)

In [91]:
from sentence_transformers import SentenceTransformer, util
import json

# Load lightweight sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Safe JSON parser
def safe_parse_json(field):
    if not field or field.strip() == "":
        return {}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {}

# Compute semantic similarity between vulnerableCode and fixedCode
code_similarity_scores = []

for sample in a:  # Use all samples
    pred = safe_parse_json(sample.get("predict", ""))
    
    vuln_code = "\n".join(pred.get("vulnerableCode", []))
    fixed_code = pred.get("fixedCode", "")

    if not vuln_code.strip() or not fixed_code.strip():
        continue

    vuln_emb = model.encode(vuln_code, convert_to_tensor=True)
    fixed_emb = model.encode(fixed_code, convert_to_tensor=True)

    score = util.cos_sim(vuln_emb, fixed_emb).item()
    code_similarity_scores.append(score)

# Report
avg_score = sum(code_similarity_scores) / len(code_similarity_scores) if code_similarity_scores else 0
print(f"Average Code-Aware Similarity (Vulnerable ↔ Fixed Code): {avg_score:.4f}")


Average Code-Aware Similarity (Vulnerable ↔ Fixed Code): 0.8646


## **Risk Assessment Alignment**  (Reason <--> Risk)

In [90]:
from sentence_transformers import SentenceTransformer, util
import json

# Load lightweight sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Safe JSON parser
def safe_parse_json(field):
    if not field or field.strip() == "":
        return {}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {}

# Compute semantic similarity between vulnerabilityReason and potentialSecurityRisk
risk_alignment_scores = []

for sample in a:  # Use all samples
    pred = safe_parse_json(sample.get("predict", ""))
    
    reason = pred.get("vulnerabilityReason", "")
    risk = pred.get("potentialSecurityRisk", "")

    if not reason.strip() or not risk.strip():
        continue

    reason_emb = model.encode(reason, convert_to_tensor=True)
    risk_emb = model.encode(risk, convert_to_tensor=True)

    score = util.cos_sim(reason_emb, risk_emb).item()
    risk_alignment_scores.append(score)

# Report
avg_score = sum(risk_alignment_scores) / len(risk_alignment_scores) if risk_alignment_scores else 0
print(f"Average Risk Assessment Alignment (Reason ↔ Risk): {avg_score:.4f}")


Average Risk Assessment Alignment (Reason ↔ Risk): 0.5405


## **OpenAI's embedding model**

In [ ]:
pip install openai

In [69]:
import os
import openai
from sklearn.metrics.pairwise import cosine_similarity

In [71]:

openai.api_key = ""


In [100]:
file_path = "generated_predictions_all_llama31.jsonl"
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
def safe_parse_json(field):
    if not field or field.strip() == "":
        return {}
    if isinstance(field, dict):
        return field
    try:
        return json.loads(field)
    except json.JSONDecodeError:
        return {}
def get_embedding(text, model="text-embedding-3-small"):
    if not text.strip():
        return None
    response = openai.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

In [106]:
all_reasons = []
index_map = []

for i, line in enumerate(lines):
    entry = json.loads(line)
    pred = safe_parse_json(entry.get("predict", ""))
    label = safe_parse_json(entry.get("label", ""))

    pred_reason = pred.get("potentialSecurityRisk", "").strip()
    label_reason = label.get("potentialSecurityRisk", "").strip()

    if pred_reason and label_reason:
        all_reasons.extend([pred_reason, label_reason])
        index_map.append(i)

# Get all embeddings in one batch call
response = openai.embeddings.create(
    input=all_reasons,
    model="text-embedding-3-small"
)
embeddings = [item.embedding for item in response.data]

# Compute similarity
similarity_scores = []
for i, idx in enumerate(index_map):
    emb_pred = embeddings[2 * i]
    emb_label = embeddings[2 * i + 1]
    score = cosine_similarity([emb_pred], [emb_label])[0][0]

    entry = json.loads(lines[idx])
    pred = safe_parse_json(entry.get("predict", ""))
    label = safe_parse_json(entry.get("label", ""))

    similarity_scores.append({
        "predict": pred.get("potentialSecurityRisk", ""),
        "label": label.get("potentialSecurityRisk", ""),
        "similarity": score
    })


In [104]:
# Print top 5 results
sum_similarity = 0
for item in similarity_scores[:]:
    # print("Prediction:", item["predict"])
    # print("Label     :", item["label"])
    # print(f"Similarity Score: {item['similarity']:.4f}")
    # print("="*50)
    sum_similarity = sum_similarity + item["similarity"]
avg_score = sum_similarity / len(similarity_scores)
print(f"Average Semantic Similarity:{avg_score:.4f}")

Average Semantic Similarity:0.8940


## Average Semantic Similarity: 77.23 between vulnerabilityReason of the prediction and the label
## Average Semantic Similarity: 73.16 between potentialSecurityRisk of the prediction and the label
## Average Semantic Similarity: 89.40 between fixedCode of the prediction and the label

## vscode :

In [61]:
a = [

  {
     "vulnerableLines": "6-10",
     "vulnerabilityReason": "This vulnerability occurs as integer underflow vulnerability. 'userBalanceToken[msg.sender]' is not checked if it is zero before sending token.",
     "potentialSecurityRisk": "An attacker may withdraw more balance than they own by send operation, this can lead to making balance of 'userBalanceToken[msg.sender]' to underflow and turn to be a very large number and enabling attacker to withdraw nearly infinite tokens.",
     "fixedCode": "function withdrawBalanceToken() public{ if(userBalanceToken[msg.sender] > 0) { if( ! (msg.sender.send(userBalanceToken[msg.sender]) ) ){ revert(); } userBalanceToken[msg.sender] = 0; } }"
  }
]


In [62]:
import json
predict = a[0]['vulnerabilityReason']
print(predict)

This vulnerability occurs as integer underflow vulnerability. 'userBalanceToken[msg.sender]' is not checked if it is zero before sending token.


In [ ]:
import json
import re

# Read from the .txt file
with open("/mnt/data/contract_4.txt", "r", encoding="utf-8") as f:
    content = f.read()

# Regex pattern to extract JSON array (used under === Time Dependency ===)
pattern = r"\[\s*{.*?}\s*]"

# Find JSON block(s) — assume the first match is the vulnerability data you want
matches = re.findall(pattern, content, flags=re.DOTALL)

if matches:
    vulnerability_data = json.loads(matches[0])

    # Save it into a JSON file
    with open("contract_4_parsed.json", "w", encoding="utf-8") as out_file:
        json.dump(vulnerability_data, out_file, indent=2)
    print(" JSON saved to 'contract_4_parsed.json'")
else:
    print(" No JSON array found in the file.")


In [63]:
from sentence_transformers import SentenceTransformer, util
import json

# Load predicted vulnerabilityReason
predict = a[0]['vulnerabilityReason'].strip()
# print("Predict:", predict)

# Load label from JSON file
with open("contract_1.json", 'r', encoding='utf-8') as file:
    label_json = json.load(file)

# Extract and strip the vulnerabilityReason from the label
label = label_json[0].get('vulnerabilityReason', '').strip()
print("Label:", label)

# Load sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Compare if both are non-empty
if predict and label:
    pred_emb = model.encode(predict, convert_to_tensor=True)
    label_emb = model.encode(label, convert_to_tensor=True)
    score = util.cos_sim(pred_emb, label_emb).item()
    print(f"Factual Consistency Score: {score:.4f}")
else:
    print("Missing vulnerabilityReason in predict or label.")


Label: The function 'withdrawBalanceToken()' allows for the possible occurrence of a re-entrancy attack by calling an external contract (i.e msg.sender.send()). Moreover, it subtracts the balance after calling the external contract, which can lead to underflows if the subtraction is larger than the balance.
Factual Consistency Score: 0.5487


## Finding the Number of vulnerabilities

In [2]:
import json
import os
import glob

In [29]:
def count_vul(folder_path):
    """
    Count total number of vuln. across all JSON files in the folder
    """
    total_vul = 0
    processed_files = 0
    failed_files = []

    json_files = glob.glob(os.path.join(folder_path, "*.json"))  # Fixed: os.path.join not os.path.json

    print(f"Found {len(json_files)} JSON files to process...")

    for file_path in json_files:  # Fixed: proper indentation for the for loop
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Count vulnerableLines in this file
            file_vul = 0  # Fixed: consistent variable naming
            if 'vulnerabilities' in data:
                file_vul = len(data['vulnerabilities'])
                total_vul += file_vul  # Fixed: consistent variable naming
            
            processed_files += 1
            print(f"File {os.path.basename(file_path)}: {file_vul} vulnerabilities")
            
        except (json.JSONDecodeError, FileNotFoundError, KeyError) as e:  # Fixed: proper indentation for except
            failed_files.append(file_path)
            print(f"Error processing {file_path}: {e}")
            continue
    
    print(f"\n=== SUMMARY ===")
    print(f"Total files processed: {processed_files}")
    print(f"Total vulnerabilities found: {total_vul}")  # Fixed: consistent variable naming
    print(f"Failed files: {len(failed_files)}")
    
    if failed_files:
        print("Failed files:", failed_files)
    
    return total_vul

# Usage
folder_path = "data/preprocessed_data/TD_ESC/LOCs"
total_bugs = count_vul(folder_path)  # Fixed: correct function name
print(f"\nTotal number of bugs across all files: {total_bugs}")


Found 72 JSON files to process...
File 100.json: 1 vulnerabilities
File 101.json: 4 vulnerabilities
File 102.json: 7 vulnerabilities
File 105.json: 5 vulnerabilities
File 106.json: 4 vulnerabilities
File 107.json: 3 vulnerabilities
File 108.json: 14 vulnerabilities
File 110.json: 12 vulnerabilities
File 111.json: 1 vulnerabilities
File 112.json: 4 vulnerabilities
File 118.json: 1 vulnerabilities
File 12.json: 1 vulnerabilities
File 124.json: 1 vulnerabilities
File 126.json: 8 vulnerabilities
File 128.json: 7 vulnerabilities
File 129.json: 5 vulnerabilities
File 130.json: 3 vulnerabilities
File 132.json: 9 vulnerabilities
File 133.json: 8 vulnerabilities
File 134.json: 3 vulnerabilities
File 136.json: 3 vulnerabilities
File 14.json: 5 vulnerabilities
File 140.json: 7 vulnerabilities
File 141.json: 2 vulnerabilities
File 143.json: 3 vulnerabilities
File 144.json: 1 vulnerabilities
File 145.json: 1 vulnerabilities
File 149.json: 2 vulnerabilities
File 15.json: 1 vulnerabilities
File 16.js

In [19]:
#### for solidifi dataset:

In [28]:
import json
import os
import glob

def count_vul(folder_path):
    """
    Count total number of vuln. across all JSON files in the folder
    """
    total_vul = 0
    processed_files = 0
    failed_files = []

    json_files = glob.glob(os.path.join(folder_path, "*.json"))

    print(f"Found {len(json_files)} JSON files to process...")

    for file_path in json_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Count vulnerabilities in this file
            # Since data is an array of vulnerability objects, just count the array length
            file_vul = len(data) if isinstance(data, list) else 0
            total_vul += file_vul
            
            processed_files += 1
            print(f"File {os.path.basename(file_path)}: {file_vul} vulnerabilities")
            
        except (json.JSONDecodeError, FileNotFoundError, KeyError) as e:
            failed_files.append(file_path)
            print(f"Error processing {file_path}: {e}")
            continue
    
    print(f"\n=== SUMMARY ===")
    print(f"Total files processed: {processed_files}")
    print(f"Total vulnerabilities found: {total_vul}")
    print(f"Failed files: {len(failed_files)}")
    
    if failed_files:
        print("Failed files:", failed_files)
    
    return total_vul

# Usage
folder_path = "data/preprocessed_data/TD_SolidiFi/LOCs"
total_bugs = count_vul(folder_path)
print(f"\nTotal number of bugs across all files: {total_bugs}")


Found 72 JSON files to process...
File 100.json: 0 vulnerabilities
File 101.json: 0 vulnerabilities
File 102.json: 0 vulnerabilities
File 105.json: 0 vulnerabilities
File 106.json: 0 vulnerabilities
File 107.json: 0 vulnerabilities
File 108.json: 0 vulnerabilities
File 110.json: 0 vulnerabilities
File 111.json: 0 vulnerabilities
File 112.json: 0 vulnerabilities
File 118.json: 0 vulnerabilities
File 12.json: 0 vulnerabilities
File 124.json: 0 vulnerabilities
File 126.json: 0 vulnerabilities
File 128.json: 0 vulnerabilities
File 129.json: 0 vulnerabilities
File 130.json: 0 vulnerabilities
File 132.json: 0 vulnerabilities
File 133.json: 0 vulnerabilities
File 134.json: 0 vulnerabilities
File 136.json: 0 vulnerabilities
File 14.json: 0 vulnerabilities
File 140.json: 0 vulnerabilities
File 141.json: 0 vulnerabilities
File 143.json: 0 vulnerabilities
File 144.json: 0 vulnerabilities
File 145.json: 0 vulnerabilities
File 149.json: 0 vulnerabilities
File 15.json: 0 vulnerabilities
File 16.json

In [31]:
import re

def parse_vulnerability_counts(file_path):
    """
    Parse the vulnerability counting file and sum totals by vulnerability type
    """
    re_total = 0
    iou_total = 0
    td_total = 0
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Find all lines with "Total vulnerabilities found: X"
        vulnerability_lines = re.findall(r'=== SUMMARY of (.+?)===.*?Total vulnerabilities found: (\d+)', content, re.DOTALL)
        
        print("Processing vulnerability counts:")
        print("-" * 50)
        
        for dataset_name, count in vulnerability_lines:
            count = int(count)
            dataset_name = dataset_name.strip()
            
            print(f"{dataset_name}: {count} vulnerabilities")
            
            # Categorize by prefix
            if dataset_name.startswith('RE_'):
                re_total += count
            elif dataset_name.startswith('IoU_'):
                iou_total += count
            elif dataset_name.startswith('TD_'):
                td_total += count
            else:
                print(f"  Warning: Unknown category for {dataset_name}")
        
        return re_total, iou_total, td_total
        
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found!")
        return 0, 0, 0
    except Exception as e:
        print(f"Error processing file: {e}")
        return 0, 0, 0

def main():
    file_path = "data/preprocessed_data/vulnerability_counting.txt"
    
    re_total, iou_total, td_total = parse_vulnerability_counts(file_path)
    
    print("\n" + "=" * 50)
    print("FINAL VULNERABILITY TOTALS BY TYPE")
    print("=" * 50)
    print(f"RE (Reentrancy):           {re_total:,}")
    print(f"IoU (Integer Over/Under):  {iou_total:,}")
    print(f"TD (Timestamp Dependency): {td_total:,}")
    print("-" * 50)
    print(f"GRAND TOTAL:               {re_total + iou_total + td_total:,}")

if __name__ == "__main__":
    main()

Processing vulnerability counts:
--------------------------------------------------
RE_smartbugs: 1088 vulnerabilities
RE_full_source2_218: 64 vulnerabilities
RE_FTSmartAudit_datasets: 31 vulnerabilities
RE_source3: 8 vulnerabilities
RE_solidifi: 1300 vulnerabilities
IoU_SolidiFi: 1414 vulnerabilities
IoU_source3: 280 vulnerabilities
IoU_FTSmartAudit_datasets: 22 vulnerabilities
TD_FTSmartAudit_datasets: 6 vulnerabilities
TD_source3: 644 vulnerabilities
TD_SolidiFi: 1388 vulnerabilities
TD_ESC: 276 vulnerabilities

FINAL VULNERABILITY TOTALS BY TYPE
RE (Reentrancy):           2,491
IoU (Integer Over/Under):  1,716
TD (Timestamp Dependency): 2,314
--------------------------------------------------
GRAND TOTAL:               6,521
